In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from nilearn.image import resample_to_img
from scipy.ndimage import rotate, gaussian_filter
from skimage.restoration import denoise_nl_means, estimate_sigma
from skimage.filters import unsharp_mask
from skimage.exposure import equalize_adapthist
import SimpleITK as sitk
import pandas as pd
import os
import glob
from tqdm import tqdm

# Assuming ROOT_DIR, CSV_PATH, AAL_PATH, LIMIT, OUTPUT_DIR are defined
# e.g., ROOT_DIR = r"D:\Projects\PROJECT_4_PROFESSOR\Sujitha_Miss\Full_dataset\PPMI_Brain_NIfTI"
# CSV_PATH = r"D:\Projects\PROJECT_4_PROFESSOR\Sujitha_Miss\Full_dataset\dataset_labels_balanced.csv"
# AAL_PATH = r"D:\Projects\PROJECT_4_PROFESSOR\Sujitha_Miss\dcm2img\MRIcroGL_windows\MRIcroGL\Resources\atlas\AAL3v1_1mm.nii.gz"
# LIMIT = 1
# OUTPUT_DIR = r"D:\Projects\PROJECT_4_PROFESSOR\Sujitha_Miss\Full_dataset\PPMI_PNG_Outputs"

def pad_to_size(image, target_size=(256, 256)):
    """Pads an image to target_size without distortion."""
    h, w = image.shape
    pad_h = max(0, target_size[0] - h)
    pad_w = max(0, target_size[1] - w)
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    return np.pad(image, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='constant', constant_values=0)

def bias_field_correction(image):
    """Applies N4ITK Bias Field Correction."""
    sitk_image = sitk.GetImageFromArray(image)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrected_image = corrector.Execute(sitk_image)
    return sitk.GetArrayFromImage(corrected_image)

def denoise_mri(image):
    """Applies Non-Local Means (NLM) Denoising."""
    sigma_est = np.mean(estimate_sigma(image))
    denoised = denoise_nl_means(image, h=1.15 * sigma_est, fast_mode=True, patch_size=5, patch_distance=3, preserve_range=True)
    return denoised

def smooth_image(image, sigma=1.0):
    """Applies Gaussian smoothing to the image."""
    return gaussian_filter(image, sigma=sigma)

def enhance_contrast(image):
    """Applies CLAHE for local contrast enhancement."""
    image = (image - np.min(image)) / (np.max(image) - np.min(image)) # Normalize to [0, 1]
    return equalize_adapthist(image, clip_limit=0.01)

def extract_and_correct_orientation(input_path, aal_path, region="Substantia_Nigra"):
    """Extracts 10 consecutive axial slices around the center (5 before, center, 4 after), applies enhancement, and pads to 256x256."""
    norm_img = nib.load(input_path)
    aal_img = nib.load(aal_path)
    aal_resampled = resample_to_img(aal_img, norm_img, interpolation="nearest", force_resample=True, copy_header=True)
    norm_data = norm_img.get_fdata()
    aal_data = aal_resampled.get_fdata()
    aal_labels = {161: "Substantia_Nigra_L", 162: "Substantia_Nigra_R"}
    matching_labels = [idx for idx, label in aal_labels.items() if region in label]
    if not matching_labels:
        raise ValueError(f"Region '{region}' not found in AAL Atlas.")
    roi_mask = np.isin(aal_data, matching_labels)
    coords = np.where(roi_mask)
    if len(coords[0]) == 0:
        raise ValueError(f"No voxels found for '{region}'.")
    center = [int(np.mean(c)) for c in coords]
    # Extract 10 consecutive axial slices around the center (5 before, center, 4 after)
    raw_slices = []
    for i in range(-10, 10):  # From -5 to +4 relative to center
        z_idx = center[2] + i
        if 0 <= z_idx < norm_data.shape[2]:
            raw_slices.append(norm_data[:, :, z_idx])
        else:
            # Pad with zeros if out of bounds
            zero_slice = np.zeros((norm_data.shape[0], norm_data.shape[1]), dtype=norm_data.dtype)
            raw_slices.append(zero_slice)
    corrected_slices = []
    for img in raw_slices:
        img = bias_field_correction(img)
        img = denoise_mri(img)
        # img = smooth_image(img, sigma=1.0)
        img = enhance_contrast(img)
        denom = np.max(img) - np.min(img) + 1e-8  # Avoid division by zero
        img = (img - np.min(img)) / denom * 255
        img = img.astype(np.uint8)
        img = np.flipud(img)
        img = rotate(img, 90, reshape=False)
        img = pad_to_size(img, target_size=(256, 256))
        corrected_slices.append(img)
    return corrected_slices  # List of 10 processed slices

def plot_10_slices(full_x, full_y, index, image_ids):
    """Plot the 10 axial slices for a given index from Full_X with labels."""
    if index >= len(full_x):
        raise ValueError(f"Index {index} out of range. Max index is {len(full_x) - 1}.")
   
    slices = full_x[index]  # Shape: (10, 256, 256, 1)
    offsets = [i for i in range(-10, 10)]  # -5 to 4
    titles = [f"Axial Offset {offset}" for offset in offsets]
    image_id = image_ids[index]
    label = "PD" if full_y[index] == 1 else "Control"  # Map 1 to PD, 0 to Control
    fig, axes = plt.subplots(2, 5, figsize=(25, 10))
    axes = axes.flatten()
    for i, (ax, title) in enumerate(zip(axes, titles)):
        ax.imshow(slices[i, :, :, 0], cmap="gray")  # Remove channel dim for plotting
        ax.set_title(f"{title}\nImage ID: {image_id}\nLabel: {label}")
        ax.axis("off")
    plt.suptitle(f"Slices for Index {index}")
    plt.tight_layout()
    plt.show()

def process_csv_and_generate_png(csv_path, aal_path, limit=None, output_dir=None):
    df = pd.read_csv(csv_path)
   
    if limit is not None:
        df = df.iloc[:limit]
    # Lists to collect all X (10 slices), Y data, and image IDs
    all_X = []
    all_Y = []
    all_image_ids = []
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing MRI Files"):
        relative_path = row['Path'].replace('Merged', 'Brain')
        input_path = os.path.join(ROOT_DIR, relative_path, "*.nii.gz")
       
        nii_files = glob.glob(input_path)
        if not nii_files:
            print(f"No .nii.gz file found in {input_path}")
            continue
        input_nii = nii_files[0]
        try:
            corrected_slices = extract_and_correct_orientation(input_nii, aal_path)
            # Determine output directory
            if output_dir is None:
                save_dir = os.path.join(ROOT_DIR, relative_path)
            else:
                save_dir = os.path.join(output_dir, relative_path)
            os.makedirs(save_dir, exist_ok=True)
            # Save PNGs
            for i, slice_img in enumerate(corrected_slices):
                offset = i - 10  # -5 to 4
                output_file = os.path.join(save_dir, f"Axial_Offset_{offset}.png")
                plt.imsave(output_file, slice_img, cmap="gray")
                print(f"Saved {output_file}")
            # Prepare X as a stack of 10 slices
            X = np.stack(corrected_slices, axis=0)  # Shape: (10, 256, 256)
            X = np.expand_dims(X, axis=-1)  # Shape: (10, 256, 256, 1)
            all_X.append(X)
            # Prepare Y (label)
            label = row['Research Group']
            Y = np.array(1 if label == "PD" else 0, dtype=np.int8)
            all_Y.append(Y)
            # Extract and save image ID (last folder in relative_path)
            image_id = os.path.basename(relative_path)  # e.g., "I10955674"
            all_image_ids.append(image_id)
        except Exception as e:
            print(f"Error processing {input_nii}: {str(e)}")
    # Save Full_X and Full_Y in OUTPUT_DIR if it exists
    if output_dir is not None and all_X and all_Y:
        Full_X = np.stack(all_X, axis=0)  # Shape: (n_samples, 10, 256, 256, 1)
        Full_Y = np.stack(all_Y, axis=0)  # Shape: (n_samples,)
        all_image_ids = np.array(all_image_ids)  # Shape: (n_samples,)
        full_x_output_file = os.path.join(output_dir, "Full_X.npy")
        full_y_output_file = os.path.join(output_dir, "Full_Y.npy")
        image_ids_output_file = os.path.join(output_dir, "Image_IDs.npy")
        np.save(full_x_output_file, Full_X)
        np.save(full_y_output_file, Full_Y)
        np.save(image_ids_output_file, all_image_ids)
        print(f"Saved {full_x_output_file}")
        print(f"Saved {full_y_output_file}")
        print(f"Saved {image_ids_output_file}")
        return Full_X, Full_Y, all_image_ids  # Return for immediate use
    return None, None, None

# Run the processing and get the arrays
Full_X, Full_Y, all_image_ids = process_csv_and_generate_png(CSV_PATH, AAL_PATH, limit=LIMIT, output_dir=OUTPUT_DIR)
# Example usage of plot_10_slices with labels
plot_10_slices(Full_X, Full_Y, 0, all_image_ids)

Processing MRI Files:   0%|                                                                    | 0/102 [00:00<?, ?it/s]

Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_-10.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_-9.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_-8.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_-7.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_-6.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_-5.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_-4.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02

Processing MRI Files:   1%|▌                                                         | 1/102 [01:03<1:46:28, 63.25s/it]

Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_1.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_2.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_3.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_4.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_5.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_6.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_58.0\I1497587\Axial_Offset_7.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\100018\3D_T2_FLAIR\2021-02-02_09_56_5

Processing MRI Files:   2%|█▏                                                        | 2/102 [02:47<2:25:59, 87.60s/it]

...

Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_12_11_12.0\I10672559\Axial_Offset_1.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_12_11_12.0\I10672559\Axial_Offset_2.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_12_11_12.0\I10672559\Axial_Offset_3.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_12_11_12.0\I10672559\Axial_Offset_4.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_12_11_12.0\I10672559\Axial_Offset_5.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_12_11_12.0\I10672559\Axial_Offset_6.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_12_11_12.0\I10672559\Axial_Offset_7.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\111278\3D_T2_FLAIR\2024-06-05_

Processing MRI Files:  19%|██████████▌                                              | 19/102 [20:29<1:31:55, 66.45s/it]

Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_58.0\I1519104\Axial_Offset_1.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_58.0\I1519104\Axial_Offset_2.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_58.0\I1519104\Axial_Offset_3.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_58.0\I1519104\Axial_Offset_4.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_58.0\I1519104\Axial_Offset_5.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_58.0\I1519104\Axial_Offset_6.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_58.0\I1519104\Axial_Offset_7.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\113369\3D_T2_FLAIR\2021-10-25_12_56_5

Processing MRI Files:  20%|███████████▏                                             | 20/102 [21:14<1:22:02, 60.03s/it]

Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_12_10_30.0\I11081808\Axial_Offset_1.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_12_10_30.0\I11081808\Axial_Offset_2.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_12_10_30.0\I11081808\Axial_Offset_3.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_12_10_30.0\I11081808\Axial_Offset_4.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_12_10_30.0\I11081808\Axial_Offset_5.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_12_10_30.0\I11081808\Axial_Offset_6.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_12_10_30.0\I11081808\Axial_Offset_7.png
Saved D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages\114534\3D_T2_FLAIR\2024-01-15_

In [3]:
# Root directory where all dataset files are stored
ROOT_DIR = r"D:\Sujitha_COMPLETE_PROJECT\2April_FULL_DATASET\SIEMENS_PPMI_BRAINONLY\SIEMENS_PPMI_BRAINONLY"

# Path to the CSV file containing the paths and research group info
CSV_PATH = r"D:\Sujitha_COMPLETE_PROJECT\2April_FULL_DATASET\siemens_filtered_df_MAPPED_BALANCED.csv" # Replace 'your_file.csv' with actual filename

# Path to the AAL atlas file (used for region segmentation)
AAL_PATH = r"D:\Sujitha_COMPLETE_PROJECT\dcm2img\MRIcroGL_windows\MRIcroGL\Resources\atlas\AAL3v1_1mm.nii.gz"

# Limit for number of rows to process from CSV (None for all rows, integer for limited rows)
LIMIT = None  # Set to an integer (e.g., 5) to process only that many rows

# Output directory for saving PNG files (if you want a centralized location instead of original dirs)
OUTPUT_DIR = r"D:\Sujitha_COMPLETE_PROJECT\SUJITHA_VERSION\pngimages"
# Note: If OUTPUT_DIR is None, PNGs will be saved in the same directory as the .nii.gz files
# Uncomment the line below to use original directories instead
# OUTPUT_DIR = None

# Create OUTPUT_DIR if it doesn't exist (only if OUTPUT_DIR is specified)
import os
if OUTPUT_DIR is not None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)